# 09 | Ablation, robustness, and portfolio evidence

## Study objective

This notebook determines which SOFC health-intelligence claims remain defensible when measurement modalities, operating regimes and repair-dependent observations are challenged explicitly.

All controlled comparisons use the same target definition, exact assessment horizons, Lin-KK-accepted EIS origins, nested physical-cell validation and persistence reference. The notebook closes the project by separating reproducible evidence from attractive but unsupported claims.

**Decision:** Which conclusions survive modality removal, data-quality screening, repair sensitivity and regime transfer, and which claims are suitable for a technical portfolio?

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

sns.set_theme(style="whitegrid", context="talk")


def find_project_root(start: Path) -> Path:
    # Find the nearest parent directory containing pyproject.toml.
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate pyproject.toml. Start Jupyter Notebook from inside the project directory."
    )


ROOT = find_project_root(Path.cwd())
PROCESSED = ROOT / "data" / "processed"
REPORT_TABLES = ROOT / "reports" / "tables"
OUTPUT_DIRECTORY = REPORT_TABLES / "ablation_and_portfolio"
FIGURE_DIRECTORY = ROOT / "reports" / "figures" / "ablation_and_portfolio"

for directory in (OUTPUT_DIRECTORY, FIGURE_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_PATH = PROCESSED / "modeling_table.parquet"
EIS_AUDIT_PATH = REPORT_TABLES / "eis_health_coupling" / "all_spectrum_lin_kk_audit.csv"

required_files = [MODEL_PATH, EIS_AUDIT_PATH]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    missing_text = "\n".join(f" - {path}" for path in missing_files)
    raise FileNotFoundError(
        "Notebook 09 requires completed outputs from the data pipeline and "
        f"Notebook 08:\n{missing_text}"
    )

HORIZONS = [1, 3, 5, 10]
RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0]
RECOMPUTE_ABLATIONS = True
RANDOM_SEED = 42

print("Python:", sys.version.split()[0])
print("Project root:", ROOT)
print("Recompute controlled ablations:", RECOMPUTE_ABLATIONS)

## 1. Evidence contract and eligible observations

Notebook 09 consumes saved evidence rather than redefining earlier conclusions. A forecast origin is eligible only when its EIS spectrum passed both Notebook 08 Lin-KK limits. A rejected EIS measurement does not invalidate independently measured SOH at the previous or future assessment.

The analysis predicts the change in composite SOH:

$$
\Delta y_{i,t,h}=y_{i,t+h}-y_{i,t},
$$

where $i$ is a physical cell, $t$ is the origin assessment and $h$ is the exact future-assessment horizon.

In [ ]:
modeling_all = (
    pd.read_parquet(MODEL_PATH).sort_values(["cell_id", "assessment_index"]).reset_index(drop=True)
)
modeling_all["regime"] = np.where(
    modeling_all["cell_id"].str.startswith("R"),
    "randomized",
    "regular",
)

lin_kk_audit = pd.read_csv(EIS_AUDIT_PATH)
required_audit_columns = {"cell_id", "assessment_index", "passes_lin_kk"}
missing_audit_columns = required_audit_columns.difference(lin_kk_audit.columns)
if missing_audit_columns:
    raise KeyError(f"Notebook 08 Lin-KK audit is missing columns: {sorted(missing_audit_columns)}")

accepted_origins = lin_kk_audit.loc[
    lin_kk_audit["passes_lin_kk"].astype(bool),
    ["cell_id", "assessment_index"],
].drop_duplicates()

eligible = modeling_all.merge(
    accepted_origins,
    on=["cell_id", "assessment_index"],
    how="inner",
    validate="one_to_one",
)

eligibility_summary = (
    modeling_all.groupby(["cell_id", "regime"], observed=True)
    .size()
    .rename("available_assessments")
    .reset_index()
    .merge(
        eligible.groupby(["cell_id", "regime"], observed=True)
        .size()
        .rename("eligible_origins")
        .reset_index(),
        on=["cell_id", "regime"],
        how="left",
        validate="one_to_one",
    )
)
eligibility_summary["excluded_origins"] = (
    eligibility_summary["available_assessments"] - eligibility_summary["eligible_origins"]
)
eligibility_summary["eligible_rate"] = (
    eligibility_summary["eligible_origins"] / eligibility_summary["available_assessments"]
)

print("Available assessment rows:", len(modeling_all))
print("Lin-KK-eligible EIS origins:", len(eligible))
display(eligibility_summary.round(4))

assert len(modeling_all) == 418
assert len(eligible) == 407

## 2. Exact-horizon forecast table

Current SOH and its previous-assessment change form the health-history baseline. Modality features are measured only at the accepted origin. Future SOH is joined by the exact assessment index rather than by positional shifting, so QC exclusions cannot shorten or distort a horizon.

In [ ]:
predictive = eligible.copy()
predictive["soh_at_origin_pct"] = predictive["soh_composite_pct"]
predictive["previous_assessment_index"] = predictive["assessment_index"] - 1

previous_soh = modeling_all[["cell_id", "assessment_index", "soh_composite_pct"]].rename(
    columns={
        "assessment_index": "previous_assessment_index",
        "soh_composite_pct": "previous_soh_pct",
    }
)
predictive = predictive.merge(
    previous_soh,
    on=["cell_id", "previous_assessment_index"],
    how="left",
    validate="many_to_one",
)
predictive["soh_delta1"] = predictive["soh_at_origin_pct"] - predictive["previous_soh_pct"]

future_soh = modeling_all[["cell_id", "assessment_index", "soh_composite_pct"]].rename(
    columns={
        "assessment_index": "future_assessment_index",
        "soh_composite_pct": "future_soh_pct",
    }
)

forecast_frames = []
for horizon in HORIZONS:
    frame = predictive.copy()
    frame["horizon"] = horizon
    frame["future_assessment_index"] = frame["assessment_index"] + horizon
    frame = frame.merge(
        future_soh,
        on=["cell_id", "future_assessment_index"],
        how="inner",
        validate="many_to_one",
    )
    frame["delta_soh_pct"] = frame["future_soh_pct"] - frame["soh_at_origin_pct"]
    if not (frame["future_assessment_index"] - frame["assessment_index"]).eq(horizon).all():
        raise ValueError("Exact forecast-horizon alignment failed.")
    forecast_frames.append(frame)

forecast_data = pd.concat(forecast_frames, ignore_index=True)
forecast_summary = forecast_data.groupby("horizon", observed=True).agg(
    rows=("delta_soh_pct", "size"),
    cells=("cell_id", "nunique"),
    first_origin=("assessment_index", "min"),
    last_origin=("assessment_index", "max"),
)

print("Forecast rows:", len(forecast_data))
display(forecast_summary)
assert len(forecast_data) == 1482
assert forecast_summary["cells"].eq(8).all()

## 3. Pre-registered modality sets

The controlled ablation uses one estimator family and one validation protocol for every feature set. This isolates the contribution of information sources from differences in model class.

- `history`: current SOH, assessment index and one-step SOH change.
- `history_plus_eis`, `history_plus_iv`, `history_plus_transient`: one modality added to history.
- `all_modalities`: all origin-time diagnostic features plus history.
- `leave_out_*`: all modalities except the named source.

Features containing target, future, RUL, EOL or interval information are excluded explicitly.

In [ ]:
HISTORY_FEATURES = [
    "soh_at_origin_pct",
    "assessment_index",
    "soh_delta1",
]


def safe_modality_columns(frame: pd.DataFrame, prefix: str) -> list[str]:
    # Select numeric origin-time features while blocking target-like names.
    forbidden_tokens = (
        "future",
        "target",
        "rul",
        "eol",
        "coverage",
        "interval",
        "prediction",
    )
    return sorted(
        column
        for column in frame.columns
        if column.startswith(prefix)
        and pd.api.types.is_numeric_dtype(frame[column])
        and not any(token in column.lower() for token in forbidden_tokens)
    )


EIS_FEATURES = safe_modality_columns(forecast_data, "eis_")
IV_FEATURES = safe_modality_columns(forecast_data, "iv_")
TRANSIENT_FEATURES = safe_modality_columns(forecast_data, "tr_")


def unique_features(*groups: list[str]) -> list[str]:
    # Combine feature groups without changing their declared order.
    return list(dict.fromkeys(feature for group in groups for feature in group))


FEATURE_SETS = {
    "history": HISTORY_FEATURES,
    "history_plus_eis": unique_features(HISTORY_FEATURES, EIS_FEATURES),
    "history_plus_iv": unique_features(HISTORY_FEATURES, IV_FEATURES),
    "history_plus_transient": unique_features(
        HISTORY_FEATURES,
        TRANSIENT_FEATURES,
    ),
    "all_modalities": unique_features(
        HISTORY_FEATURES,
        EIS_FEATURES,
        IV_FEATURES,
        TRANSIENT_FEATURES,
    ),
    "leave_out_eis": unique_features(
        HISTORY_FEATURES,
        IV_FEATURES,
        TRANSIENT_FEATURES,
    ),
    "leave_out_iv": unique_features(
        HISTORY_FEATURES,
        EIS_FEATURES,
        TRANSIENT_FEATURES,
    ),
    "leave_out_transient": unique_features(
        HISTORY_FEATURES,
        EIS_FEATURES,
        IV_FEATURES,
    ),
}

feature_contract = pd.DataFrame(
    [
        {
            "feature_set": name,
            "feature_count": len(features),
            "features": ", ".join(features),
        }
        for name, features in FEATURE_SETS.items()
    ]
)
feature_contract.to_csv(
    OUTPUT_DIRECTORY / "ablation_feature_contract.csv",
    index=False,
)

print("EIS features:", len(EIS_FEATURES))
print("IV features:", len(IV_FEATURES))
print("Transient features:", len(TRANSIENT_FEATURES))
display(feature_contract[["feature_set", "feature_count"]])

assert all(FEATURE_SETS.values())

## 4. Nested leave-one-cell-out ablation

For each horizon and held-out physical cell, ridge regularization is selected using leave-one-cell-out validation inside the remaining seven cells. The outer cell is never available to feature scaling, imputation, tuning or fitting.

For cell $i$, model skill relative to persistence is

$$
S_{i,h}=1-\frac{\mathrm{MAE}_{i,h}^{\mathrm{model}}}
{\mathrm{MAE}_{i,h}^{\mathrm{persistence}}}.
$$

Positive skill indicates improvement; negative skill indicates harm.

In [ ]:
def make_ridge(alpha: float) -> Pipeline:
    return Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", RobustScaler()),
            ("ridge", Ridge(alpha=alpha)),
        ]
    )


def tune_alpha_by_cells(
    training: pd.DataFrame,
    predictors: list[str],
) -> tuple[float, pd.DataFrame]:
    # Select alpha using macro-cell MAE inside the outer training cells.
    records = []
    validation_cells = sorted(training["cell_id"].unique())
    if len(validation_cells) < 2:
        raise ValueError("Inner cell validation requires at least two cells.")

    for alpha in RIDGE_ALPHAS:
        cell_maes = []
        for validation_cell in validation_cells:
            inner_train = training.loc[training["cell_id"] != validation_cell]
            inner_validation = training.loc[training["cell_id"] == validation_cell]
            overlap = set(inner_train["cell_id"]).intersection(inner_validation["cell_id"])
            if overlap:
                raise ValueError(f"Inner cell leakage detected: {sorted(overlap)}")

            estimator = make_ridge(alpha)
            estimator.fit(
                inner_train[predictors],
                inner_train["delta_soh_pct"],
            )
            prediction = estimator.predict(inner_validation[predictors])
            cell_maes.append(
                mean_absolute_error(
                    inner_validation["delta_soh_pct"],
                    prediction,
                )
            )

        records.append(
            {
                "alpha": alpha,
                "inner_macro_cell_mae": float(np.mean(cell_maes)),
                "inner_worst_cell_mae": float(np.max(cell_maes)),
            }
        )

    tuning = pd.DataFrame(records).sort_values(
        ["inner_macro_cell_mae", "inner_worst_cell_mae", "alpha"]
    )
    return float(tuning.iloc[0]["alpha"]), tuning


def nested_cell_evaluation(
    data: pd.DataFrame,
    feature_sets: dict[str, list[str]],
    experiment: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Run one controlled nested cell-isolated experiment.
    metrics = []
    predictions = []
    tuning_frames = []
    split_records = []

    for horizon in HORIZONS:
        horizon_data = data.loc[data["horizon"] == horizon]
        for held_out_cell in sorted(horizon_data["cell_id"].unique()):
            train = horizon_data.loc[horizon_data["cell_id"] != held_out_cell]
            test = horizon_data.loc[horizon_data["cell_id"] == held_out_cell]
            overlap = set(train["cell_id"]).intersection(test["cell_id"])
            split_records.append(
                {
                    "experiment": experiment,
                    "horizon": horizon,
                    "held_out_cell": held_out_cell,
                    "training_cells": ",".join(sorted(train["cell_id"].unique())),
                    "overlapping_cells": ",".join(sorted(overlap)),
                    "leakage_violation": bool(overlap),
                }
            )
            if overlap:
                raise ValueError(f"Outer cell leakage detected: {sorted(overlap)}")

            actual = test["delta_soh_pct"].to_numpy(dtype=float)
            persistence_mae = mean_absolute_error(actual, np.zeros(len(test)))

            for feature_set, predictors in feature_sets.items():
                best_alpha, tuning = tune_alpha_by_cells(train, predictors)
                tuning_frames.append(
                    tuning.assign(
                        experiment=experiment,
                        horizon=horizon,
                        held_out_cell=held_out_cell,
                        feature_set=feature_set,
                    )
                )
                estimator = make_ridge(best_alpha)
                estimator.fit(train[predictors], train["delta_soh_pct"])
                predicted = estimator.predict(test[predictors])
                model_mae = mean_absolute_error(actual, predicted)
                model_rmse = mean_squared_error(actual, predicted) ** 0.5
                metrics.append(
                    {
                        "experiment": experiment,
                        "horizon": horizon,
                        "cell_id": held_out_cell,
                        "regime": test["regime"].iloc[0],
                        "feature_set": feature_set,
                        "features": len(predictors),
                        "best_alpha": best_alpha,
                        "observations": len(test),
                        "mae": model_mae,
                        "rmse": model_rmse,
                        "bias": float(np.mean(predicted - actual)),
                        "persistence_mae": persistence_mae,
                        "skill_vs_persistence": (
                            1 - model_mae / persistence_mae if persistence_mae > 0 else np.nan
                        ),
                    }
                )
                for row_number, (_, row) in enumerate(test.iterrows()):
                    predictions.append(
                        {
                            "experiment": experiment,
                            "horizon": horizon,
                            "cell_id": held_out_cell,
                            "regime": row["regime"],
                            "assessment_index": row["assessment_index"],
                            "future_assessment_index": row["future_assessment_index"],
                            "feature_set": feature_set,
                            "actual_delta_soh_pct": actual[row_number],
                            "predicted_delta_soh_pct": predicted[row_number],
                        }
                    )

    return (
        pd.DataFrame(metrics),
        pd.DataFrame(predictions),
        pd.concat(tuning_frames, ignore_index=True),
        pd.DataFrame(split_records),
    )

In [ ]:
metric_cache = OUTPUT_DIRECTORY / "controlled_modality_ablation_metrics.csv"
prediction_cache = OUTPUT_DIRECTORY / "controlled_modality_ablation_predictions.csv"
tuning_cache = OUTPUT_DIRECTORY / "controlled_modality_ablation_tuning.csv"
split_cache = OUTPUT_DIRECTORY / "controlled_modality_ablation_split_audit.csv"

cache_files = [metric_cache, prediction_cache, tuning_cache, split_cache]
if RECOMPUTE_ABLATIONS:
    (
        ablation_metrics,
        ablation_predictions,
        ablation_tuning,
        ablation_split_audit,
    ) = nested_cell_evaluation(
        forecast_data,
        FEATURE_SETS,
        experiment="controlled_modality_ablation",
    )
    for frame, path in zip(
        [
            ablation_metrics,
            ablation_predictions,
            ablation_tuning,
            ablation_split_audit,
        ],
        cache_files,
        strict=True,
    ):
        frame.to_csv(path, index=False)
else:
    missing_cache = [path for path in cache_files if not path.exists()]
    if missing_cache:
        raise FileNotFoundError(
            "Ablation cache is missing. Set RECOMPUTE_ABLATIONS = True for the first complete run."
        )
    (
        ablation_metrics,
        ablation_predictions,
        ablation_tuning,
        ablation_split_audit,
    ) = [pd.read_csv(path) for path in cache_files]

expected_metric_rows = len(HORIZONS) * 8 * len(FEATURE_SETS)
expected_prediction_rows = len(forecast_data) * len(FEATURE_SETS)
ablation_leakage_violations = int(ablation_split_audit["leakage_violation"].sum())

print("Metric rows:", len(ablation_metrics))
print("Prediction rows:", len(ablation_predictions))
print("Tuning rows:", len(ablation_tuning))
print("Outer splits audited:", len(ablation_split_audit))
print("Cell-leakage violations:", ablation_leakage_violations)

assert len(ablation_metrics) == expected_metric_rows
assert len(ablation_predictions) == expected_prediction_rows
assert ablation_leakage_violations == 0

## 5. Modality-ablation results

Macro-cell MAE gives every physical cell equal influence regardless of trajectory length. A modality is useful only when its contribution is directionally stable across cells and horizons, not merely when pooled error improves.

In [ ]:
ablation_summary = (
    ablation_metrics.groupby(["horizon", "feature_set"], observed=True)
    .agg(
        cells=("cell_id", "nunique"),
        macro_cell_mae=("mae", "mean"),
        macro_cell_rmse=("rmse", "mean"),
        macro_bias=("bias", "mean"),
        persistence_macro_mae=("persistence_mae", "mean"),
        median_cell_skill=("skill_vs_persistence", "median"),
        minimum_cell_skill=("skill_vs_persistence", "min"),
        cells_beating_persistence=(
            "skill_vs_persistence",
            lambda values: int((values > 0).sum()),
        ),
    )
    .reset_index()
)
ablation_summary["macro_skill_vs_persistence"] = 1 - (
    ablation_summary["macro_cell_mae"] / ablation_summary["persistence_macro_mae"]
)
ablation_summary.to_csv(
    OUTPUT_DIRECTORY / "controlled_modality_ablation_summary.csv",
    index=False,
)

print("Controlled modality performance")
display(ablation_summary.sort_values(["horizon", "macro_cell_mae"]).round(4))

skill_heatmap = ablation_summary.pivot(
    index="feature_set",
    columns="horizon",
    values="macro_skill_vs_persistence",
).reindex(FEATURE_SETS)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(
    skill_heatmap,
    annot=True,
    fmt="+.2f",
    center=0,
    cmap="RdYlGn",
    linewidths=0.5,
    cbar_kws={"label": "Macro-cell skill versus persistence"},
    ax=ax,
)
ax.set_title("Controlled modality ablation by forecast horizon", fontweight="bold")
ax.set_xlabel("Forecast horizon")
ax.set_ylabel("Feature set")
plt.tight_layout()
ablation_figure = FIGURE_DIRECTORY / "controlled_modality_ablation.png"
fig.savefig(ablation_figure, dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved to:", ablation_figure)

In [ ]:
all_modality_mae = ablation_metrics.loc[
    ablation_metrics["feature_set"] == "all_modalities",
    ["horizon", "cell_id", "mae"],
].rename(columns={"mae": "all_modalities_mae"})

contribution_frames = []
for omitted_set, modality in {
    "leave_out_eis": "EIS",
    "leave_out_iv": "IV",
    "leave_out_transient": "transient",
}.items():
    omitted = ablation_metrics.loc[
        ablation_metrics["feature_set"] == omitted_set,
        ["horizon", "cell_id", "mae"],
    ].rename(columns={"mae": "leave_out_mae"})
    comparison = all_modality_mae.merge(
        omitted,
        on=["horizon", "cell_id"],
        how="inner",
        validate="one_to_one",
    )
    comparison["modality"] = modality
    comparison["relative_value_of_modality"] = 1 - (
        comparison["all_modalities_mae"] / comparison["leave_out_mae"]
    )
    contribution_frames.append(comparison)

modality_contribution = pd.concat(contribution_frames, ignore_index=True)
modality_contribution_summary = (
    modality_contribution.groupby(["horizon", "modality"], observed=True)
    .agg(
        cells=("cell_id", "nunique"),
        median_relative_value=("relative_value_of_modality", "median"),
        mean_relative_value=("relative_value_of_modality", "mean"),
        minimum_relative_value=("relative_value_of_modality", "min"),
        cells_helped=(
            "relative_value_of_modality",
            lambda values: int((values > 0).sum()),
        ),
    )
    .reset_index()
)
modality_contribution.to_csv(
    OUTPUT_DIRECTORY / "leave_one_modality_out_cell_results.csv",
    index=False,
)
modality_contribution_summary.to_csv(
    OUTPUT_DIRECTORY / "leave_one_modality_out_summary.csv",
    index=False,
)

print("Incremental value in the all-modality model")
display(modality_contribution_summary.round(4))

## 6. Repaired-source sensitivity

Notebook 01 repaired `N5/IV_curves/IV13.mat` and `IV22.mat`. The original missing or unreadable measurements cannot be reconstructed, so this is an exclusion sensitivity test rather than a comparison with unknown original values.

The challenge removes every forecast pair whose origin or future target is N5 assessment 13 or 22, then reruns the history and all-modality models. A conclusion is repair-sensitive if its direction changes materially after those pairs are removed.

In [ ]:
REPAIR_CELL = "N5"
REPAIR_ASSESSMENTS = {13, 22}

repair_dependent_pair = forecast_data["cell_id"].eq(REPAIR_CELL) & (
    forecast_data["assessment_index"].isin(REPAIR_ASSESSMENTS)
    | forecast_data["future_assessment_index"].isin(REPAIR_ASSESSMENTS)
)
repair_exclusion_data = forecast_data.loc[~repair_dependent_pair].copy()
repair_feature_sets = {
    name: FEATURE_SETS[name]
    for name in [
        "history",
        "history_plus_iv",
        "all_modalities",
        "leave_out_iv",
    ]
}

repair_metric_cache = OUTPUT_DIRECTORY / "repair_exclusion_metrics.csv"
repair_split_cache = OUTPUT_DIRECTORY / "repair_exclusion_split_audit.csv"

if RECOMPUTE_ABLATIONS:
    (
        repair_exclusion_metrics,
        _,
        _,
        repair_split_audit,
    ) = nested_cell_evaluation(
        repair_exclusion_data,
        repair_feature_sets,
        experiment="exclude_n5_repaired_assessments",
    )
    repair_exclusion_metrics.to_csv(repair_metric_cache, index=False)
    repair_split_audit.to_csv(repair_split_cache, index=False)
else:
    repair_exclusion_metrics = pd.read_csv(repair_metric_cache)
    repair_split_audit = pd.read_csv(repair_split_cache)

full_repair_comparator = ablation_metrics.loc[
    ablation_metrics["feature_set"].isin(repair_feature_sets),
    ["horizon", "cell_id", "feature_set", "mae"],
].rename(columns={"mae": "full_data_mae"})
repair_sensitivity = full_repair_comparator.merge(
    repair_exclusion_metrics[["horizon", "cell_id", "feature_set", "mae"]].rename(
        columns={"mae": "repair_exclusion_mae"}
    ),
    on=["horizon", "cell_id", "feature_set"],
    how="inner",
    validate="one_to_one",
)
repair_sensitivity["relative_mae_change_after_exclusion"] = (
    repair_sensitivity["repair_exclusion_mae"] / repair_sensitivity["full_data_mae"] - 1
)
repair_summary = (
    repair_sensitivity.groupby(["horizon", "feature_set"], observed=True)
    .agg(
        cells=("cell_id", "nunique"),
        median_relative_mae_change=(
            "relative_mae_change_after_exclusion",
            "median",
        ),
        maximum_absolute_relative_mae_change=(
            "relative_mae_change_after_exclusion",
            lambda values: float(values.abs().max()),
        ),
    )
    .reset_index()
)
repair_sensitivity.to_csv(
    OUTPUT_DIRECTORY / "n5_repair_sensitivity_cell_results.csv",
    index=False,
)
repair_summary.to_csv(
    OUTPUT_DIRECTORY / "n5_repair_sensitivity_summary.csv",
    index=False,
)

print("Forecast pairs removed:", int(repair_dependent_pair.sum()))
print(
    "Repair-sensitivity leakage violations:",
    int(repair_split_audit["leakage_violation"].sum()),
)
display(repair_summary.round(4))

## 7. Cross-regime transfer diagnostic

This deliberately difficult test trains on one operating regime and evaluates the other. It is not equivalent to field validation. Training on randomized-redox data is especially uncertain because only R1 and R2 are available.

In [ ]:
# Cross-regime transfer diagnostic
#
# This is a diagnostic stress test, not a deployment validation.
# Hyperparameters are selected using only training-regime cells.
# Final performance is calculated as macro-cell MAE:
#   1. calculate MAE independently for every test cell
#   2. average the cell MAEs with equal cell weight

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

TRANSFER_FEATURE_SETS = {
    "history": FEATURE_SETS["history"],
    "all_modalities": FEATURE_SETS["all_modalities"],
}

TRANSFER_DIRECTIONS = [
    ("regular", "randomized"),
    ("randomized", "regular"),
]

ALPHA_GRID = np.logspace(-3, 3, 13)


def make_ridge_pipeline(alpha):
    """Construct the complete preprocessing and Ridge model pipeline."""
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("ridge", Ridge(alpha=float(alpha))),
        ]
    )


def calculate_cell_metrics(evaluation_table):
    """
    Calculate model and persistence MAE independently for each cell.

    Returns one row per physical test cell.
    """
    rows = []

    for cell_id, cell_table in evaluation_table.groupby(
        "cell_id",
        observed=True,
        sort=False,
    ):
        rows.append(
            {
                "cell_id": cell_id,
                "observations": len(cell_table),
                "model_mae": mean_absolute_error(
                    cell_table["actual"],
                    cell_table["predicted"],
                ),
                "persistence_mae": mean_absolute_error(
                    cell_table["actual"],
                    cell_table["persistence"],
                ),
            }
        )

    return pd.DataFrame(rows)


def tune_alpha_by_training_cell(
    training_table,
    predictors,
    target_column,
):
    """
    Select Ridge alpha using leave-one-training-cell-out validation.

    Every physical cell used for validation is absent from the corresponding
    model-fitting data. Alpha selection therefore remains cell-leakage safe.
    """
    training_cells = sorted(training_table["cell_id"].unique())
    tuning_rows = []
    split_rows = []

    if len(training_cells) < 2:
        raise ValueError("At least two training cells are required for transfer tuning.")

    for alpha in ALPHA_GRID:
        validation_cell_maes = []

        for validation_cell in training_cells:
            inner_train = training_table.loc[training_table["cell_id"] != validation_cell].copy()

            inner_validation = training_table.loc[
                training_table["cell_id"] == validation_cell
            ].copy()

            train_cells = set(inner_train["cell_id"])
            validation_cells = set(inner_validation["cell_id"])
            overlap = train_cells.intersection(validation_cells)

            split_rows.append(
                {
                    "split_level": "transfer_inner",
                    "alpha": float(alpha),
                    "validation_cell": validation_cell,
                    "training_cells": len(train_cells),
                    "validation_cells": len(validation_cells),
                    "leakage_violation": len(overlap) > 0,
                }
            )

            if overlap:
                raise RuntimeError(
                    f"Cell leakage detected during transfer-model tuning: {sorted(overlap)}"
                )

            model = make_ridge_pipeline(alpha)

            model.fit(
                inner_train[predictors],
                inner_train[target_column],
            )

            validation_prediction = model.predict(inner_validation[predictors])

            validation_cell_maes.append(
                mean_absolute_error(
                    inner_validation[target_column],
                    validation_prediction,
                )
            )

        tuning_rows.append(
            {
                "alpha": float(alpha),
                "macro_cell_mae": float(np.mean(validation_cell_maes)),
                "minimum_cell_mae": float(np.min(validation_cell_maes)),
                "maximum_cell_mae": float(np.max(validation_cell_maes)),
                "training_cells": len(training_cells),
            }
        )

    tuning_table = pd.DataFrame(tuning_rows)

    # Use stronger regularization as the deterministic tie-breaker.
    best_row = tuning_table.sort_values(
        ["macro_cell_mae", "alpha"],
        ascending=[True, False],
    ).iloc[0]

    return (
        float(best_row["alpha"]),
        tuning_table,
        pd.DataFrame(split_rows),
    )


transfer_metric_rows = []
transfer_prediction_tables = []
transfer_tuning_tables = []
transfer_split_audits = []

for train_regime, test_regime in TRANSFER_DIRECTIONS:
    for horizon in HORIZONS:
        horizon_table = forecast_data.loc[forecast_data["horizon"] == horizon].copy()

        training_table = horizon_table.loc[horizon_table["regime"] == train_regime].copy()

        test_table = horizon_table.loc[horizon_table["regime"] == test_regime].copy()

        training_cells = sorted(training_table["cell_id"].unique())
        test_cells = sorted(test_table["cell_id"].unique())

        outer_overlap = set(training_cells).intersection(test_cells)

        transfer_split_audits.append(
            pd.DataFrame(
                [
                    {
                        "split_level": "transfer_outer",
                        "train_regime": train_regime,
                        "test_regime": test_regime,
                        "horizon": horizon,
                        "training_cells": len(training_cells),
                        "test_cells": len(test_cells),
                        "leakage_violation": len(outer_overlap) > 0,
                    }
                ]
            )
        )

        if outer_overlap:
            raise RuntimeError(
                "Cell leakage detected between transfer training and test "
                f"sets: {sorted(outer_overlap)}"
            )

        for feature_set_name, predictors in TRANSFER_FEATURE_SETS.items():
            predictors = list(predictors)

            target_column = "future_soh_pct"

            required_columns = ["cell_id", "regime", "horizon", target_column] + predictors

            missing_columns = [
                column for column in required_columns if column not in horizon_table.columns
            ]

            if missing_columns:
                raise KeyError(f"Missing columns for {feature_set_name}: {missing_columns}")

            best_alpha, tuning_table, inner_audit = tune_alpha_by_training_cell(
                training_table=training_table,
                predictors=predictors,
                target_column=target_column,
            )

            tuning_table = tuning_table.assign(
                train_regime=train_regime,
                test_regime=test_regime,
                horizon=horizon,
                feature_set=feature_set_name,
            )
            transfer_tuning_tables.append(tuning_table)

            inner_audit = inner_audit.assign(
                train_regime=train_regime,
                test_regime=test_regime,
                horizon=horizon,
                feature_set=feature_set_name,
            )
            transfer_split_audits.append(inner_audit)

            final_model = make_ridge_pipeline(best_alpha)

            final_model.fit(
                training_table[predictors],
                training_table[target_column],
            )

            predicted = final_model.predict(test_table[predictors])

            test_evaluation = pd.DataFrame(
                {
                    "cell_id": test_table["cell_id"].to_numpy(),
                    "actual": test_table[target_column].to_numpy(),
                    "predicted": predicted,
                    # SOH at forecast origin is the persistence prediction.
                    "persistence": test_table["soh_at_origin_pct"].to_numpy(),
                },
                index=test_table.index,
            )

            cell_metrics = calculate_cell_metrics(test_evaluation)

            macro_cell_mae = cell_metrics["model_mae"].mean()
            persistence_macro_mae = cell_metrics["persistence_mae"].mean()

            if persistence_macro_mae > 0:
                skill_vs_persistence = 1.0 - macro_cell_mae / persistence_macro_mae
            else:
                skill_vs_persistence = np.nan

            cell_metrics = cell_metrics.assign(
                train_regime=train_regime,
                test_regime=test_regime,
                horizon=horizon,
                feature_set=feature_set_name,
                best_alpha=best_alpha,
                cell_skill_vs_persistence=(
                    1.0 - cell_metrics["model_mae"] / cell_metrics["persistence_mae"]
                ),
            )

            transfer_metric_rows.append(
                {
                    "train_regime": train_regime,
                    "test_regime": test_regime,
                    "horizon": horizon,
                    "feature_set": feature_set_name,
                    "training_cells": len(training_cells),
                    "test_cells": len(test_cells),
                    "test_observations": len(test_table),
                    "best_alpha": best_alpha,
                    "macro_cell_mae": macro_cell_mae,
                    "persistence_macro_cell_mae": (persistence_macro_mae),
                    "skill_vs_persistence": skill_vs_persistence,
                    "median_cell_skill": cell_metrics["cell_skill_vs_persistence"].median(),
                    "minimum_cell_skill": cell_metrics["cell_skill_vs_persistence"].min(),
                    "cells_beating_persistence": int(
                        (cell_metrics["cell_skill_vs_persistence"] > 0).sum()
                    ),
                }
            )

            prediction_table = test_evaluation.assign(
                train_regime=train_regime,
                test_regime=test_regime,
                horizon=horizon,
                feature_set=feature_set_name,
                best_alpha=best_alpha,
            )

            transfer_prediction_tables.append(prediction_table)


cross_regime_metrics = pd.DataFrame(transfer_metric_rows)

cross_regime_predictions = pd.concat(
    transfer_prediction_tables,
    ignore_index=True,
)

cross_regime_tuning = pd.concat(
    transfer_tuning_tables,
    ignore_index=True,
)

cross_regime_split_audit = pd.concat(
    transfer_split_audits,
    ignore_index=True,
)

cross_regime_metrics.to_csv(
    OUTPUT_DIRECTORY / "cross_regime_transfer_metrics.csv",
    index=False,
)
cross_regime_predictions.to_csv(
    OUTPUT_DIRECTORY / "cross_regime_transfer_predictions.csv",
    index=False,
)
cross_regime_tuning.to_csv(
    OUTPUT_DIRECTORY / "cross_regime_transfer_tuning.csv",
    index=False,
)
cross_regime_split_audit.to_csv(
    OUTPUT_DIRECTORY / "cross_regime_transfer_split_audit.csv",
    index=False,
)

transfer_leakage_violations = int(cross_regime_split_audit["leakage_violation"].sum())

print(f"Transfer metric rows: {len(cross_regime_metrics)}")
print(f"Transfer prediction rows: {len(cross_regime_predictions)}")
print(f"Transfer splits audited: {len(cross_regime_split_audit)}")
print(f"Transfer cell-leakage violations: {transfer_leakage_violations}")

if transfer_leakage_violations != 0:
    raise RuntimeError("Cross-regime transfer analysis contains cell leakage.")

display(
    cross_regime_metrics[
        [
            "train_regime",
            "test_regime",
            "horizon",
            "feature_set",
            "training_cells",
            "test_cells",
            "test_observations",
            "best_alpha",
            "macro_cell_mae",
            "persistence_macro_cell_mae",
            "skill_vs_persistence",
            "median_cell_skill",
            "minimum_cell_skill",
            "cells_beating_persistence",
        ]
    ].round(4)
)


# Plot transfer skill using separate panels.
#
# Separate panels prevent catastrophic randomized-to-regular failures from
# compressing the useful regular-to-randomized values into an unreadable
# colour range.

plot_table = cross_regime_metrics.copy()

plot_table["transfer_direction"] = plot_table["train_regime"] + " to " + plot_table["test_regime"]

transfer_directions = [
    "regular to randomized",
    "randomized to regular",
]

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(13, 4.8),
    constrained_layout=True,
)

for axis, direction in zip(
    axes,
    transfer_directions,
    strict=True,
):
    direction_table = (
        plot_table.loc[plot_table["transfer_direction"] == direction]
        .pivot(
            index="feature_set",
            columns="horizon",
            values="skill_vs_persistence",
        )
        .reindex(
            index=["history", "all_modalities"],
            columns=HORIZONS,
        )
    )

    # Plot values are clipped only for colour scaling.
    # The annotations retain the exact calculated values.
    colour_values = direction_table.clip(lower=-1.0, upper=1.0)

    sns.heatmap(
        colour_values,
        annot=direction_table,
        fmt="+.2f",
        cmap="RdYlGn",
        center=0,
        vmin=-1,
        vmax=1,
        linewidths=0.5,
        linecolor="white",
        cbar=axis is axes[-1],
        cbar_kws=({"label": "Macro-cell skill versus persistence"} if axis is axes[-1] else None),
        ax=axis,
    )

    axis.set_title(direction.replace(" to ", " → "))
    axis.set_xlabel("Forecast horizon")
    axis.set_ylabel("Feature set")

fig.suptitle(
    "Cross-regime transfer diagnostic",
    fontsize=14,
    fontweight="bold",
)

from pathlib import Path

# Resolve the project root whether Jupyter was started from the project root
# or from the notebooks directory.
current_directory = Path.cwd().resolve()

if current_directory.name.lower() == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

cross_regime_figure_directory = project_root / "reports" / "figures" / "ablation_and_portfolio"

cross_regime_figure_directory.mkdir(
    parents=True,
    exist_ok=True,
)

cross_regime_figure_path = cross_regime_figure_directory / "cross_regime_transfer.png"

fig.savefig(
    cross_regime_figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print(f"Figure saved to: {cross_regime_figure_path}")


# Engineering interpretation generated directly from the results.

regular_to_randomized = cross_regime_metrics.loc[cross_regime_metrics["train_regime"].eq("regular")]

randomized_to_regular = cross_regime_metrics.loc[
    cross_regime_metrics["train_regime"].eq("randomized")
]

print("\nEngineering interpretation")
print(
    "Regular-to-randomized transfer is evaluated using six training cells "
    "and two independent randomized test cells."
)
print(
    "Randomized-to-regular transfer is substantially less reliable because "
    "only two randomized cells are available for training. The result is "
    "therefore primarily a covariate-shift and limited-replication stress "
    "test."
)
print(
    "A positive transfer skill means that the model improves on persistence "
    "after every test cell receives equal weight. A negative value means "
    "that persistence is the safer forecast."
)

if (
    randomized_to_regular.loc[
        randomized_to_regular["feature_set"].eq("all_modalities"),
        "skill_vs_persistence",
    ]
    < 0
).any():
    print(
        "The all-modality model fails in at least one randomized-to-regular "
        "transfer condition. This rejects regime-invariant deployment of "
        "the current high-dimensional feature representation."
    )

### Engineering decision

The cross-regime transfer test rejects the assumption that a single SOH
forecasting model can be transferred safely between regular and randomized
redox operation. Regular-to-randomized transfer is approximately equivalent
to persistence at short horizons and becomes unreliable at horizon 10.
Randomized-to-regular transfer fails at every horizon, reflecting both strong
covariate shift and the inadequate representation provided by only two
randomized training cells.

The high-dimensional all-modality model is less transferable than the
history-only model and produces physically implausible extrapolation under
regime shift. Consequently, the current models must remain regime-specific.
A regime-invariant model would require substantially more randomized cells,
overlapping operating conditions, explicit distribution-shift controls and
prospective external validation.

## 8. Uncertainty and lifetime evidence synthesis

Notebook 07 established that exact RUL at the selected 80% SOH threshold is not identifiable because all eight cells are right-censored. It also showed that marginal conformal coverage can conceal severe cell-horizon undercoverage. Notebook 09 carries those findings forward without manufacturing a lifetime estimate from uncrossed trajectories.

In [ ]:
# Import canonical decision registers from completed notebooks.
#
# Do not recursively search the reports directory because Notebook 09 writes
# its own combined decision-register files there. Recursive discovery would
# cause Notebook 09 to ingest its previous outputs and duplicate decisions.

from pathlib import Path

current_directory = Path.cwd().resolve()

if current_directory.name.lower() == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

canonical_decision_files = [
    (project_root / "reports" / "tables" / "uncertainty" / "notebook07_decision_register.csv"),
    (
        project_root
        / "reports"
        / "tables"
        / "eis_health_coupling"
        / "notebook08_decision_register.csv"
    ),
]

existing_decision_files = [path for path in canonical_decision_files if path.exists()]

missing_decision_files = [path for path in canonical_decision_files if not path.exists()]

print(
    "Canonical decision-register files found:",
    len(existing_decision_files),
)

for path in existing_decision_files:
    print("  FOUND:", path.relative_to(project_root))

for path in missing_decision_files:
    print("  MISSING:", path.relative_to(project_root))


prior_decision_frames = []

for decision_file in existing_decision_files:
    decision_table = pd.read_csv(decision_file)

    required_columns = {
        "decision",
        "evidence",
        "status",
    }

    missing_columns = required_columns.difference(decision_table.columns)

    if missing_columns:
        raise ValueError(
            f"{decision_file.name} is missing required columns: {sorted(missing_columns)}"
        )

    decision_table = decision_table.copy()

    decision_table["source"] = str(decision_file.relative_to(project_root))

    if "screening_rule" not in decision_table.columns:
        decision_table["screening_rule"] = pd.NA

    prior_decision_frames.append(
        decision_table[
            [
                "decision",
                "evidence",
                "status",
                "source",
                "screening_rule",
            ]
        ]
    )


if prior_decision_frames:
    prior_decision_register = pd.concat(
        prior_decision_frames,
        ignore_index=True,
    )

    # Remove genuinely identical decisions while retaining decisions whose
    # evidence or status changed between notebooks.
    prior_decision_register = prior_decision_register.drop_duplicates(
        subset=[
            "decision",
            "evidence",
            "status",
            "screening_rule",
        ],
        keep="last",
    ).reset_index(drop=True)
else:
    prior_decision_register = pd.DataFrame(
        columns=[
            "decision",
            "evidence",
            "status",
            "source",
            "screening_rule",
        ]
    )


print(
    "Unique imported prior decisions:",
    len(prior_decision_register),
)

display(prior_decision_register)

## 9. Final evidence gate

The release decision is based on four separate questions: predictive value, stability across physical cells, sensitivity to data choices and scientific scope. Passing one question cannot compensate automatically for failing another.

In [ ]:
# Notebook 09 final decision synthesis
#
# This cell:
# 1. identifies the complete 32-row controlled-ablation summary,
# 2. separates minimum-MAE ranking from engineering selection,
# 3. validates the corrected macro-cell cross-regime analysis,
# 4. creates the final decision register,
# 5. saves all final tables.

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# =========================================================================
# 1. Resolve project and output directories
# =========================================================================

current_directory = Path.cwd().resolve()

if current_directory.name.lower() == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

final_table_directory = project_root / "reports" / "tables" / "ablation_and_portfolio"

final_table_directory.mkdir(
    parents=True,
    exist_ok=True,
)


# =========================================================================
# 2. Locate the complete controlled-ablation summary
# =========================================================================

required_summary_columns = {
    "horizon",
    "feature_set",
    "macro_cell_mae",
    "macro_skill_vs_persistence",
    "minimum_cell_skill",
}

required_feature_sets = {
    "history",
    "history_plus_eis",
    "history_plus_transient",
    "all_modalities",
}

required_horizons = {1, 3, 5, 10}

controlled_summary_candidates = []


def normalize_feature_set_name(value):
    """Normalize feature-set names for reliable matching."""
    return str(value).strip().lower().replace("+", "_plus_").replace(" ", "_").replace("-", "_")


for variable_name, variable_value in list(globals().items()):
    if not isinstance(variable_value, pd.DataFrame):
        continue

    if not required_summary_columns.issubset(variable_value.columns):
        continue

    candidate = variable_value.copy()

    candidate["horizon"] = pd.to_numeric(
        candidate["horizon"],
        errors="coerce",
    )

    candidate["feature_set"] = (
        candidate["feature_set"]
        .map(normalize_feature_set_name)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    candidate_feature_sets = set(candidate["feature_set"].dropna())

    candidate_horizons = set(candidate["horizon"].dropna().astype(int))

    candidate_combinations = candidate[["horizon", "feature_set"]].dropna().drop_duplicates()

    if (
        required_feature_sets.issubset(candidate_feature_sets)
        and required_horizons.issubset(candidate_horizons)
        and len(candidate_combinations) >= 32
    ):
        controlled_summary_candidates.append(
            {
                "name": variable_name,
                "table": candidate,
                "combinations": len(candidate_combinations),
                "rows": len(candidate),
            }
        )


if not controlled_summary_candidates:
    print("DataFrames containing some required summary columns:")

    for variable_name, variable_value in list(globals().items()):
        if not isinstance(variable_value, pd.DataFrame):
            continue

        matched_columns = required_summary_columns.intersection(variable_value.columns)

        if matched_columns:
            print(
                f"  {variable_name}: "
                f"{len(variable_value)} rows; "
                f"matched columns={sorted(matched_columns)}"
            )

    raise NameError(
        "The complete controlled-ablation summary was not found. "
        "Rerun the controlled modality ablation summary cell "
        "that generates all 8 feature sets across 4 horizons."
    )


controlled_summary_candidates.sort(
    key=lambda item: (
        item["combinations"],
        item["rows"],
    ),
    reverse=True,
)

selected_candidate = controlled_summary_candidates[0]

selected_summary_name = selected_candidate["name"]

controlled_summary = selected_candidate["table"].copy()

print(
    "Complete controlled-ablation summary selected:",
    selected_summary_name,
)

print(
    "Original rows:",
    len(controlled_summary),
)

print(
    "Unique feature-set and horizon combinations:",
    selected_candidate["combinations"],
)


# =========================================================================
# 3. Clean and validate the controlled summary
# =========================================================================

controlled_summary["horizon"] = pd.to_numeric(
    controlled_summary["horizon"],
    errors="raise",
).astype(int)

controlled_summary["feature_set"] = (
    controlled_summary["feature_set"]
    .map(normalize_feature_set_name)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)

numeric_summary_columns = [
    "macro_cell_mae",
    "macro_skill_vs_persistence",
    "minimum_cell_skill",
]

for column in numeric_summary_columns:
    controlled_summary[column] = pd.to_numeric(
        controlled_summary[column],
        errors="raise",
    )

controlled_summary = (
    controlled_summary[
        [
            "horizon",
            "feature_set",
            "macro_cell_mae",
            "macro_skill_vs_persistence",
            "minimum_cell_skill",
        ]
    ]
    .drop_duplicates(
        subset=[
            "horizon",
            "feature_set",
        ],
        keep="last",
    )
    .sort_values(
        [
            "horizon",
            "feature_set",
        ]
    )
    .reset_index(drop=True)
)

available_combinations = set(
    zip(
        controlled_summary["horizon"],
        controlled_summary["feature_set"],
        strict=True,
    )
)

expected_combinations = {
    (horizon, feature_set)
    for horizon in sorted(required_horizons)
    for feature_set in [
        "history",
        "history_plus_eis",
        "history_plus_iv",
        "history_plus_transient",
        "all_modalities",
        "leave_out_eis",
        "leave_out_iv",
        "leave_out_transient",
    ]
}

missing_combinations = expected_combinations - available_combinations

if missing_combinations:
    raise ValueError(
        "The controlled-ablation summary is incomplete. "
        f"Missing combinations: {sorted(missing_combinations)}"
    )

print(
    "Validated controlled combinations:",
    len(available_combinations),
)


# =========================================================================
# 4. Numerically best feature set by macro-cell MAE
# =========================================================================

best_by_macro_mae = (
    controlled_summary.sort_values(
        [
            "horizon",
            "macro_cell_mae",
            "feature_set",
        ]
    )
    .groupby(
        "horizon",
        as_index=False,
        observed=True,
    )
    .first()[
        [
            "horizon",
            "feature_set",
            "macro_cell_mae",
            "macro_skill_vs_persistence",
            "minimum_cell_skill",
        ]
    ]
)

display(Markdown("### Numerically best controlled feature set by macro-cell MAE"))

display(best_by_macro_mae.round(4))


# =========================================================================
# 5. Engineering model recommendation
# =========================================================================

recommended_feature_sets = {
    1: "history",
    3: "history",
    5: "history_plus_eis",
    10: "history_plus_eis",
}

recommendation_reasons = {
    1: (
        "History is recommended because it beats persistence across "
        "all eight cells and avoids the severe worst-cell failure "
        "of history_plus_transient."
    ),
    3: ("History gives the lowest macro-cell MAE with the simplest predictor representation."),
    5: (
        "History_plus_eis gives strong positive macro skill and "
        "positive skill for every held-out cell."
    ),
    10: (
        "History_plus_eis gives the strongest long-horizon macro "
        "skill and positive skill for every held-out cell."
    ),
}

engineering_recommendation_rows = []

for horizon, feature_set_name in recommended_feature_sets.items():
    matching_rows = controlled_summary.loc[
        controlled_summary["horizon"].eq(horizon)
        & controlled_summary["feature_set"].eq(feature_set_name)
    ]

    if len(matching_rows) != 1:
        raise ValueError(
            "Expected exactly one controlled-ablation row for "
            f"horizon={horizon}, "
            f"feature_set={feature_set_name}; "
            f"found {len(matching_rows)}."
        )

    selected_row = matching_rows.iloc[0]

    engineering_recommendation_rows.append(
        {
            "horizon": horizon,
            "recommended_feature_set": (feature_set_name),
            "macro_cell_mae": selected_row["macro_cell_mae"],
            "macro_skill_vs_persistence": selected_row["macro_skill_vs_persistence"],
            "minimum_cell_skill": selected_row["minimum_cell_skill"],
            "engineering_reason": (recommendation_reasons[horizon]),
        }
    )

engineering_recommendations = pd.DataFrame(engineering_recommendation_rows)

display(Markdown("### Engineering model recommendation by horizon"))

display(engineering_recommendations.round(4))


# =========================================================================
# 6. Extract controlled-ablation evidence
# =========================================================================


def get_controlled_result(
    horizon,
    feature_set_name,
):
    """Return one validated controlled-ablation result."""
    rows = controlled_summary.loc[
        controlled_summary["horizon"].eq(horizon)
        & controlled_summary["feature_set"].eq(feature_set_name)
    ]

    if len(rows) != 1:
        raise ValueError(
            "Expected one controlled result for "
            f"horizon={horizon}, "
            f"feature_set={feature_set_name}; "
            f"found {len(rows)}."
        )

    return rows.iloc[0]


horizon_1_history = get_controlled_result(
    1,
    "history",
)

horizon_1_transient = get_controlled_result(
    1,
    "history_plus_transient",
)

horizon_3_history = get_controlled_result(
    3,
    "history",
)

horizon_5_eis = get_controlled_result(
    5,
    "history_plus_eis",
)

horizon_10_eis = get_controlled_result(
    10,
    "history_plus_eis",
)

all_modality_rows = controlled_summary.loc[controlled_summary["feature_set"].eq("all_modalities")]

positive_all_modality_horizons = int(all_modality_rows["macro_skill_vs_persistence"].gt(0).sum())


# =========================================================================
# 7. Validate corrected cross-regime transfer results
# =========================================================================

required_transfer_columns = {
    "train_regime",
    "test_regime",
    "horizon",
    "feature_set",
    "skill_vs_persistence",
    "cells_beating_persistence",
}

if "cross_regime_metrics" not in globals():
    raise NameError(
        "cross_regime_metrics was not found. Run the corrected cross-regime transfer cell first."
    )

missing_transfer_columns = required_transfer_columns - set(cross_regime_metrics.columns)

if missing_transfer_columns:
    raise ValueError(f"cross_regime_metrics is missing columns: {sorted(missing_transfer_columns)}")

transfer_results = cross_regime_metrics.copy()

transfer_results["horizon"] = pd.to_numeric(
    transfer_results["horizon"],
    errors="raise",
).astype(int)

transfer_results["feature_set"] = (
    transfer_results["feature_set"]
    .map(normalize_feature_set_name)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)

transfer_results = (
    transfer_results.drop_duplicates(
        subset=[
            "train_regime",
            "test_regime",
            "horizon",
            "feature_set",
        ],
        keep="last",
    )
    .sort_values(
        [
            "train_regime",
            "test_regime",
            "horizon",
            "feature_set",
        ]
    )
    .reset_index(drop=True)
)

expected_transfer_rows = 16

if len(transfer_results) != expected_transfer_rows:
    raise ValueError(
        "Unexpected number of cross-regime results. "
        f"Expected {expected_transfer_rows}, "
        f"found {len(transfer_results)}."
    )

positive_transfer_count = int(transfer_results["skill_vs_persistence"].gt(0).sum())

regular_to_randomized = transfer_results.loc[
    transfer_results["train_regime"].eq("regular")
    & transfer_results["test_regime"].eq("randomized")
]

randomized_to_regular = transfer_results.loc[
    transfer_results["train_regime"].eq("randomized")
    & transfer_results["test_regime"].eq("regular")
]

positive_regular_to_randomized = int(regular_to_randomized["skill_vs_persistence"].gt(0).sum())

positive_randomized_to_regular = int(randomized_to_regular["skill_vs_persistence"].gt(0).sum())

print(
    "\nPositive cross-regime configurations:",
    f"{positive_transfer_count} of {len(transfer_results)}",
)

print(
    "Regular-to-randomized positive configurations:",
    f"{positive_regular_to_randomized} of {len(regular_to_randomized)}",
)

print(
    "Randomized-to-regular positive configurations:",
    f"{positive_randomized_to_regular} of {len(randomized_to_regular)}",
)

if positive_transfer_count != 4:
    raise RuntimeError(
        "Expected 4 positive transfer configurations from "
        "the corrected macro-cell analysis, but found "
        f"{positive_transfer_count}. Rerun the corrected "
        "cross-regime cell."
    )

if positive_randomized_to_regular != 0:
    raise RuntimeError("Expected zero randomized-to-regular configurations to beat persistence.")


# =========================================================================
# 8. Validate leakage-audit evidence
# =========================================================================

if "cross_regime_split_audit" not in globals():
    raise NameError(
        "cross_regime_split_audit was not found. "
        "Run the corrected cross-regime transfer cell first."
    )

transfer_splits_audited = len(cross_regime_split_audit)

transfer_leakage_violations = int(cross_regime_split_audit["leakage_violation"].sum())

if transfer_leakage_violations != 0:
    raise RuntimeError("Cross-regime transfer leakage violations were found.")

print(
    "Transfer splits audited:",
    transfer_splits_audited,
)

print(
    "Transfer cell-leakage violations:",
    transfer_leakage_violations,
)


# =========================================================================
# 9. Create final Notebook 09 decision register
# =========================================================================

notebook09_decision_register = pd.DataFrame(
    [
        {
            "decision": ("Controlled modality ablation"),
            "evidence": (
                "Eight feature sets were evaluated at four "
                "forecast horizons using eight held-out physical "
                "cells with nested cell-isolated tuning."
            ),
            "status": ("COMPLETED WITH NESTED CELL ISOLATION"),
        },
        {
            "decision": ("All-modality predictive value"),
            "evidence": (
                f"The all-modality model produced positive "
                f"macro skill at "
                f"{positive_all_modality_horizons} of 4 "
                "horizons but was not the engineering-recommended "
                "model at any horizon."
            ),
            "status": ("NOT UNIFORMLY SUPPORTED"),
        },
        {
            "decision": ("Short-horizon model selection"),
            "evidence": (
                "At horizon 1, history_plus_transient reduced "
                "macro-cell MAE from "
                f"{horizon_1_history['macro_cell_mae']:.4f} "
                "to "
                f"{horizon_1_transient['macro_cell_mae']:.4f}, "
                "but its minimum cell skill was "
                f"{horizon_1_transient['minimum_cell_skill']:.4f} "
                "compared with "
                f"{horizon_1_history['minimum_cell_skill']:.4f} "
                "for history."
            ),
            "status": ("HISTORY RECOMMENDED FOR WORST-CELL ROBUSTNESS"),
        },
        {
            "decision": ("Horizon-3 model selection"),
            "evidence": (
                "The history model achieved macro-cell MAE "
                f"{horizon_3_history['macro_cell_mae']:.4f}, "
                "macro skill "
                f"{horizon_3_history['macro_skill_vs_persistence']:.4f} "
                "and minimum cell skill "
                f"{horizon_3_history['minimum_cell_skill']:.4f}."
            ),
            "status": ("HISTORY RECOMMENDED"),
        },
        {
            "decision": ("Medium-horizon EIS value"),
            "evidence": (
                "At horizon 5, history_plus_eis achieved "
                "macro-cell MAE "
                f"{horizon_5_eis['macro_cell_mae']:.4f}, "
                "macro skill "
                f"{horizon_5_eis['macro_skill_vs_persistence']:.4f} "
                "and minimum cell skill "
                f"{horizon_5_eis['minimum_cell_skill']:.4f}."
            ),
            "status": ("QUALITY-SCREENED EIS SUPPORTED AT HORIZON 5"),
        },
        {
            "decision": ("Long-horizon EIS value"),
            "evidence": (
                "At horizon 10, history_plus_eis achieved "
                "macro-cell MAE "
                f"{horizon_10_eis['macro_cell_mae']:.4f}, "
                "macro skill "
                f"{horizon_10_eis['macro_skill_vs_persistence']:.4f} "
                "and minimum cell skill "
                f"{horizon_10_eis['minimum_cell_skill']:.4f}."
            ),
            "status": ("QUALITY-SCREENED EIS SUPPORTED AT HORIZON 10"),
        },
        {
            "decision": ("Stable modality contribution"),
            "evidence": (
                "Leave-one-modality-out contribution changes "
                "with modality and forecast horizon. Additional "
                "modalities do not consistently reduce "
                "held-out-cell forecast error."
            ),
            "status": ("REPORT BY MODALITY AND HORIZON"),
        },
        {
            "decision": ("N5 repair sensitivity"),
            "evidence": (
                "Excluding 16 forecast pairs associated with "
                "repaired N5 IV assessments produced small "
                "median MAE changes. The maximum absolute "
                "cell-level relative MAE change was approximately "
                "7.04%."
            ),
            "status": ("EXCLUSION SENSITIVITY, NOT ORIGINAL-DATA RECOVERY"),
        },
        {
            "decision": ("Cross-regime transfer"),
            "evidence": (
                f"Positive macro-cell skill occurred in only "
                f"{positive_transfer_count} of "
                f"{len(transfer_results)} "
                "direction-horizon-model combinations. "
                "Randomized-to-regular transfer achieved "
                f"positive skill in "
                f"{positive_randomized_to_regular} of "
                f"{len(randomized_to_regular)} combinations."
            ),
            "status": ("NOT SUPPORTED; REGIME-SPECIFIC MODELLING REQUIRED"),
        },
        {
            "decision": ("Cross-regime leakage audit"),
            "evidence": (
                f"{transfer_splits_audited} transfer splits "
                "were audited with "
                f"{transfer_leakage_violations} physical-cell "
                "leakage violations."
            ),
            "status": ("LEAKAGE AUDIT PASSED"),
        },
        {
            "decision": ("Uncertainty and RUL"),
            "evidence": (
                "Notebook 07 found all cells right-censored "
                "at 80% SOH, pooled interval undercoverage "
                "and no observed end-of-life event."
            ),
            "status": ("LOWER-BOUND LIFETIME AND SOH INTERVALS ONLY"),
        },
        {
            "decision": ("Release scope"),
            "evidence": (
                "Evidence is limited to eight laboratory cells, "
                "assessment-index time, regime-dependent "
                "behaviour and no external field validation."
            ),
            "status": ("PORTFOLIO-GRADE OFFLINE STUDY; NOT DEPLOYMENT-VALIDATED"),
        },
    ]
)

display(Markdown("### Notebook 09 final decision register"))

display(notebook09_decision_register)


# =========================================================================
# 10. Final engineering decision
# =========================================================================

engineering_decision_text = """
### Final engineering decision

Model selection must be horizon specific. SOH history is the safer choice at
forecast horizons 1 and 3. Although transient features slightly reduce average
MAE at horizon 1, their severe worst-cell degradation makes them unsuitable as
the preferred engineering model.

The quality-screened eight-feature EIS representation provides stable
cross-cell forecasting value at horizons 5 and 10. This does not contradict
Notebook 08. Notebook 08 tested a compact resistance-based EIS representation,
whereas Notebook 09 evaluates a broader quality-screened EIS feature set.

Combining every modality is not automatically beneficial. IV and transient
features introduce redundancy, noise and unstable interactions in some
held-out cells. Model complexity must therefore be justified separately at
each forecast horizon.

Cross-regime transfer is not supported. Regular-to-randomized transfer is weak
and inconsistent, while randomized-to-regular transfer fails for every tested
configuration. The current models must remain regime specific.

The completed work is a portfolio-grade offline laboratory study. It supports
quality-screened EIS interpretation, leakage-safe cross-cell comparison and
horizon-specific model selection. It does not support exact RUL estimation,
regime-independent forecasting or field deployment.
"""

display(Markdown(engineering_decision_text))


# =========================================================================
# 11. Save final tables
# =========================================================================

decision_register_path = final_table_directory / "notebook09_decision_register.csv"

best_macro_path = final_table_directory / "best_feature_set_by_macro_mae.csv"

engineering_recommendation_path = final_table_directory / "engineering_model_recommendations.csv"

transfer_metrics_path = final_table_directory / "cross_regime_transfer_metrics.csv"

notebook09_decision_register.to_csv(
    decision_register_path,
    index=False,
)

best_by_macro_mae.to_csv(
    best_macro_path,
    index=False,
)

engineering_recommendations.to_csv(
    engineering_recommendation_path,
    index=False,
)

transfer_results.to_csv(
    transfer_metrics_path,
    index=False,
)

print("\nSaved final Notebook 09 tables:")

for saved_path in [
    decision_register_path,
    best_macro_path,
    engineering_recommendation_path,
    transfer_metrics_path,
]:
    print(" ", saved_path)

## 10. Portfolio communication

The project story should lead with the engineering decisions, validation design
and limitations, not the number of algorithms tested.

### Defensible project summary

Built a reproducible, physics-informed SOFC health-intelligence workflow for
418 aligned EIS, IV and transient assessments across eight physical cells.
Defined a transparent composite SOH target, screened all EIS spectra using
Lin-KK consistency, enforced nested leave-one-cell-out validation, quantified
cell-level conformal uncertainty and tested the incremental forecasting value
of each diagnostic modality beyond observed SOH history.

Of the 418 EIS spectra, 407 passed the project-specific Lin-KK acceptance
criteria of residual RMSE no greater than 1% and maximum absolute residual no
greater than 5%. The remaining 11 spectra were retained in the raw data but
excluded from downstream EIS interpretation and EIS-based forecast origins.

### Verified forecasting decisions

Model selection is horizon specific.

- At horizon 1, the history model is recommended. It achieved a macro-cell MAE
  of 0.5392 SOH percentage points, 9.21% skill relative to persistence and
  positive skill for all eight held-out cells.

- At horizon 3, the history model is recommended. It achieved a macro-cell MAE
  of 1.0082 and 11.00% macro skill, although the minimum cell skill was
  -7.07%. This remaining worst-cell weakness must be reported.

- At horizon 5, the history-plus-EIS model is recommended. It achieved a
  macro-cell MAE of 0.9826, 29.00% macro skill and positive skill for every
  held-out cell.

- At horizon 10, the history-plus-EIS model is recommended. It achieved a
  macro-cell MAE of 1.1471, 44.30% macro skill and positive skill for every
  held-out cell.

The numerically lowest horizon-1 MAE was obtained using history plus transient
features, at 0.5352. However, its minimum cell skill was -96.24%. The history
model was therefore selected because its small average-MAE disadvantage was
outweighed by substantially safer worst-cell behaviour.

### Interpretation of EIS value

Notebook 08 showed that a compact resistance-based EIS representation did not
provide consistent incremental forecasting value beyond current SOH and SOH
history.

Notebook 09 does not invalidate that result. It shows that a broader
quality-screened eight-feature EIS representation provides stable incremental
value at forecast horizons 5 and 10. The engineering conclusion is therefore
that EIS value depends on feature representation, quality screening and
forecast horizon.

The concurrent associations between polarization-related EIS indicators and
SOH or IV performance remain exploratory evidence from eight laboratory cells.
They support diagnostic relevance but do not establish a unique degradation
mechanism or causal relationship.

### Modality-ablation decision

Combining all diagnostic modalities is not automatically beneficial. The
all-modality model produced positive average skill at three of four horizons
but was not the engineering-recommended model at any horizon.

Leave-one-modality-out results also changed with forecast horizon. EIS was most
useful at longer horizons, while IV and transient features introduced
redundancy or unstable interactions in some held-out cells. Each modality must
therefore justify its inclusion through leakage-safe, cell-level validation.

The N5 repaired-IV exclusion analysis removed 16 forecast pairs. Median
performance changes remained small, while the maximum observed absolute
cell-level relative MAE change was approximately 7.04%. This supports
robustness to exclusion of the repaired assessments, but it does not recover
or authenticate the original missing measurements.

### Cross-regime transfer decision

Cross-regime forecasting is not supported.

Only 4 of 16 direction-horizon-model transfer configurations achieved positive
macro-cell skill. Regular-to-randomized transfer was positive in 4 of 8
configurations, while randomized-to-regular transfer was positive in 0 of 8
configurations.

The transfer analysis audited 840 inner and outer splits and detected zero
physical-cell leakage violations. The failure is therefore an observed
generalization limitation, not evidence of cell overlap between training and
test data.

The randomized regime contains only two cells, which is insufficient to support
a general model for the six regular cells. The current models must remain
regime specific until substantially more independent randomized and regular
cells are available.

### Uncertainty and RUL decision

Exact 80% SOH remaining useful life is not estimable because none of the eight
cells reached the defined end-of-life threshold. All lifetime observations are
right-censored.

The uncertainty analysis may therefore report lower-bound lifetime evidence and
SOH forecast intervals, but it must not report validated operating-hour RUL.
The pooled interval coverage of 0.878 was below the 0.90 acceptance target, and
the worst cell-horizon coverage requires explicit disclosure.

### Equivalent-circuit interpretation

The one-arc equivalent circuit is retained as a numerical sensitivity
benchmark. It is not a uniquely identified electrochemical mechanism. Agreement
between packages for selected resistance estimates does not prove that the
circuit topology uniquely represents the underlying physical processes.

### Claims supported after Notebook 09

- The 418 spectra share a common acquisition structure, and 407 passed the
  defined Lin-KK interpretation screen.
- SOH history contains forecastable cross-cell structure beyond persistence.
- History is the safer forecasting representation at horizons 1 and 3.
- Quality-screened EIS features add stable cross-cell forecasting value at
  horizons 5 and 10.
- EIS value depends on feature representation and forecast horizon.
- More diagnostic modalities do not necessarily produce a better forecast.
- Polarization-related EIS indicators are associated with concurrent SOH and
  performance degradation, especially under regular operation.
- Model behaviour depends strongly on the redox regime.
- The N5 repair-exclusion sensitivity does not materially change the central
  portfolio conclusions.
- All validation and transfer experiments use physical-cell isolation with zero
  detected leakage violations.

### Claims that remain prohibited

- Field-deployment readiness.
- Validated operating-hour forecasting.
- Exact 80% SOH RUL estimation.
- Universal transfer across regular and randomized redox regimes.
- A regime-independent all-modality model.
- A unique electrochemical mechanism inferred from one equivalent circuit.
- Causal degradation claims based only on concurrent association.
- Performance claims based only on pooled observations.
- Performance claims produced using random row-level train-test splits.
- Selecting only the most favourable cell, model or forecast horizon.

### Resume-ready project statement

Developed a physics-informed SOFC health-intelligence workflow across 418
multimodal assessments from eight laboratory cells, including all-spectrum
Lin-KK quality screening, interpretable SOH construction, nested
leave-one-cell-out forecasting, modality ablation and uncertainty analysis.
Quality-screened EIS features improved macro-cell forecast skill to 29.0% at
horizon 5 and 44.3% at horizon 10, with positive skill for every held-out cell
at both horizons. The study also identified strong redox-regime dependence,
rejected cross-regime deployment and explicitly separated diagnostic
association from causal or field-validation claims.

## Reproducibility outputs

Tables are saved under `reports/tables/ablation_and_portfolio/`:

- `ablation_feature_contract.csv`
- `controlled_modality_ablation_metrics.csv`
- `controlled_modality_ablation_predictions.csv`
- `controlled_modality_ablation_tuning.csv`
- `controlled_modality_ablation_split_audit.csv`
- `controlled_modality_ablation_summary.csv`
- `leave_one_modality_out_cell_results.csv`
- `leave_one_modality_out_summary.csv`
- `repair_exclusion_metrics.csv`
- `repair_exclusion_split_audit.csv`
- `n5_repair_sensitivity_cell_results.csv`
- `n5_repair_sensitivity_summary.csv`
- `cross_regime_transfer_metrics.csv`
- `cross_regime_transfer_predictions.csv`
- `cross_regime_transfer_tuning.csv`
- `cross_regime_transfer_split_audit.csv`
- `best_feature_set_by_macro_mae.csv`
- `engineering_model_recommendations.csv`
- `notebook09_decision_register.csv`

Figures are saved under `reports/figures/ablation_and_portfolio/`.

The notebook is complete when **Restart and Run All** produces all declared artifacts, audits every outer split with zero physical-cell overlap, retains all eight cells at every horizon, and finishes with claims that agree with the evidence rather than overriding it.